# Notebook 1 — Data Gathering and Manipulation (NumPy & Pandas)

**Duration:** 75 minutes  
**Audience:** Undergraduates with intermediate Python, introductory chemistry knowledge  
**Goal:** locate, load, inspect, clean, and perform basic feature engineering on the Fe-Redox dataset so it is ready for analysis and model training.


## Learning objectives

- Find the dataset files in the repository and load them into pandas.
- Inspect data types, summary statistics, and identify missing values.
- Detect and handle outliers using the IQR method.
- Compute simple structural features (e.g., Fe–ligand bond lengths) from coordinate data.
- Save a cleaned CSV for downstream analysis.

---

## Part 1 — Locating and understanding the data

### 1.1 Repository layout and data location

Start by inspecting the repository to find likely data folders (examples: `data/`, `datasets/`, `raw/`, `structures/`). Use the Jupyter file browser or a short shell listing in a terminal cell.

### 1.2 Loading the main dataset

We prefer a CSV entrypoint when available and fall back to a JSON/other parser if not. The pattern below is resilient: try to read a CSV named `data.csv` or `dataset.csv`, otherwise search for `*.json`.



In [ ]:

# Standard imports
import os
from pathlib import Path
import pandas as pd

# Try common CSV locations, otherwise fallback to searching for a JSON file
candidates = ["data.csv", "dataset.csv", "data/data.csv", "datasets/data.csv"]
repo_root = Path.cwd()

df = None
for c in candidates:
    p = repo_root / c
    if p.exists():
        df = pd.read_csv(p)
        print(f"Loaded CSV: {p}")
        break

if df is None:
    # Fallback: look for a small number of json files and pick the largest plausible one
    json_files = list(repo_root.rglob("*.json"))
    if json_files:
        # prefer files with 'data' in the name
        json_files.sort(key=lambda p: ("data" in p.name, p.stat().st_size), reverse=True)
        df = pd.read_json(json_files[0], orient="records")
        print(f"Loaded JSON: {json_files[0]}")

if df is None:
    raise RuntimeError("Could not find dataset CSV or JSON. Verify the data files are in the repository.")




### 1.3 Alternative structural parsing (XYZ-like files)

If raw structure files are present (e.g., `.xyz`, `.traj`), you can extract coordinate arrays using a small parser. We keep this helper focused and safe: it parses simple XYZ files with atomic labels and 3 floats per atom line.



In [ ]:

import numpy as np

def parse_xyz_file(path):
    """Parse a minimal XYZ-like file into a list of (atom, x, y, z) arrays.

    Returns: list of tuples (symbols: list[str], coords: np.ndarray of shape (N,3))
    """
    symbols = []
    coords = []
    with open(path, "r") as fh:
        lines = [l.strip() for l in fh if l.strip()]
    # Skip header if the first line is an integer atom count
    start = 0
    if lines and lines[0].split()[0].isdigit():
        start = 1
    for line in lines[start:]:
        parts = line.split()
        if len(parts) < 4:
            continue
        symbols.append(parts[0])
        coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
    if not coords:
        return None
    return symbols, np.asarray(coords)




---

## Part 2 — Data inspection with pandas

### 2.1 First look

Show the first few rows and the shape of the data. Use `display()` in Jupyter so wide tables render nicely.



In [ ]:

from IPython.display import display

if 'df' not in globals():
    raise RuntimeError("Load the dataset in the previous cell before running this one.")

print("Shape:", df.shape)
display(df.head())




### 2.2 Data types and summary statistics

Inspect dtypes and a numeric summary. This helps identify columns that need casting or contain unexpected values.



In [ ]:

print(df.dtypes)
display(df.describe(include='all').T)




### 2.3 Column completeness and uniqueness

Check non-null counts and unique values for categorical columns. This gives a quick picture of missingness and label cardinality.



In [ ]:

nulls = df.isnull().sum().sort_values(ascending=False)
print(nulls[nulls>0])

cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
for c in cat_cols:
    print(c, "-> unique:", df[c].nunique())




### 2.4 Physical meaning of common columns

The dataset typically contains physical and computed features for Fe complexes. A short reference:

| Column | Meaning | Typical range / notes |
|---|---:|---|
| `fe_oxidation_state` | Formal oxidation state of Fe | integers, e.g., 2, 3 |
| `redox_potential` | Experimental redox potential (V vs reference) | continuous, dataset-dependent |
| `fe_x, fe_y, fe_z` | Cartesian coordinates of the Fe atom | Angstroms |
| `ligand_coords` | Serialized coordinate block or file pointer | requires parsing |
| `homo_energy`, `lumo_energy` | Orbital energies (eV) from DFT | continuous |

Adjust this table to match the actual column names in your dataset.

### 2.5 Identifying the target and candidate features

Use heuristics to identify the redox target and numeric features for modeling.



In [ ]:

target_keywords = ['redox', 'potential', 'e_redox', 'e0']
feature_keywords = ['bond', 'length', 'distance', 'homo', 'lumo', 'coord', 'charge', 'dipole']

possible_targets = [c for c in df.columns if any(t in c.lower() for t in target_keywords)]
print('Possible targets:', possible_targets)

numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
print('Numeric columns (candidate features):', numeric_cols)




---

## Part 3 — Data cleaning

### 3.1 Handling missing values

Strategy: prefer dropping rows with missing target values; for features, either impute (mean/median) or flag missingness via an indicator column.



In [ ]:

# Drop rows missing the primary target
if possible_targets:
    target = possible_targets[0]
    before = len(df)
    df = df[df[target].notnull()].copy()
    print(f"Dropped {before - len(df)} rows with missing target '{target}'")
else:
    raise RuntimeError("No target column identified. Please set 'target' manually.")

# Impute numeric features with median to be robust to outliers
num_cols = df.select_dtypes(include=["number"]).columns.tolist()
num_cols.remove(target) if target in num_cols else None
for c in num_cols:
    if df[c].isnull().any():
        med = df[c].median()
        df[f"{c}_was_missing"] = df[c].isnull()
        df[c].fillna(med, inplace=True)
        print(f"Imputed {c} with median = {med}")




### 3.2 Outlier detection with the IQR method

We use the interquartile range (IQR) to flag extreme values. Recall the definitions:

IQR and outlier bounds:

$$
\text{IQR} = Q_3 - Q_1
$$

A sample $x$ is considered an outlier if:

$$
x < Q_1 - 1.5\,\text{IQR} \quad \text{or} \quad x > Q_3 + 1.5\,\text{IQR}
$$

Apply this per numeric column and optionally remove the flagged rows or save a Boolean mask.



In [ ]:

import numpy as np

outlier_mask = pd.DataFrame(False, index=df.index, columns=num_cols)

for c in num_cols:
    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df[c] < lower) | (df[c] > upper)
    outlier_mask[c] = mask
    print(f"{c}: outliers {mask.sum()} (lower={lower}, upper={upper})")

# Aggregate outlier flag (True if any feature flags an outlier)
df['is_outlier_any'] = outlier_mask.any(axis=1)
print('Total rows with any outlier:', df['is_outlier_any'].sum())

# Optional: remove extreme outliers (commented by default)
# df = df[~df['is_outlier_any']].copy()




Produce a simple boxplot figure of selected columns to visualize outliers.



In [ ]:

import matplotlib.pyplot as plt

sel = [c for c in num_cols if c in ['redox_potential', 'homo_energy', 'lumo_energy']][:6]
if not sel:
    sel = num_cols[:6]

fig, axes = plt.subplots(1, len(sel), figsize=(4*len(sel), 4), squeeze=False)
axes = np.atleast_1d(axes).flatten()
for ax, c in zip(axes, sel):
    ax.boxplot(df[c].dropna())
    ax.set_title(c)
# Hide unused axes
for ax in axes[len(sel):]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig('outlier_boxplots.png', dpi=150)
plt.show()




### 3.3 Feature engineering: compute Fe–ligand bond lengths

We compute Euclidean distances between Fe coordinates and ligand atom coordinates. In 3D, the Euclidean distance is:

$$
d_{ij} = \lVert \mathbf{r}_i - \mathbf{r}_j \rVert_2 = \sqrt{\sum_{k=1}^{3} (r_{i,k} - r_{j,k})^2}
$$

Implement a helper that extracts Fe coordinates from a row and computes bond-length statistics (min, max, mean) to be used as features.



In [ ]:

import math

# Example convention: store fe coordinates in columns fe_x, fe_y, fe_z

def compute_fe_bond_lengths(row, ligand_column='ligand_coords'):
    # Expect fe_x, fe_y, fe_z to exist
    if not all(k in row.index for k in ['fe_x', 'fe_y', 'fe_z']):
        return pd.Series({'fe_bond_min': np.nan, 'fe_bond_mean': np.nan, 'fe_bond_max': np.nan})
    fe_coord = np.array([row['fe_x'], row['fe_y'], row['fe_z']], dtype=float)
    # ligand_coords may be a serialized block; here we assume it's a list of coordinates or path to an xyz
    ligand_coords = row.get(ligand_column, None)
    if ligand_coords is None or ligand_coords is np.nan:
        return pd.Series({'fe_bond_min': np.nan, 'fe_bond_mean': np.nan, 'fe_bond_max': np.nan})
    # If ligand_coords is a string and points to a file, try parsing
    if isinstance(ligand_coords, str) and os.path.exists(ligand_coords):
        parsed = parse_xyz_file(ligand_coords)
        if parsed is None:
            return pd.Series({'fe_bond_min': np.nan, 'fe_bond_mean': np.nan, 'fe_bond_max': np.nan})
        _, coords = parsed
    elif isinstance(ligand_coords, (list, tuple)):
        coords = np.asarray(ligand_coords, dtype=float)
    elif isinstance(ligand_coords, np.ndarray):
        coords = ligand_coords
    else:
        # Try to parse a simple serialized string like "[(x,y,z), ...]"
        try:
            coords = np.array(eval(ligand_coords), dtype=float)
        except Exception:
            return pd.Series({'fe_bond_min': np.nan, 'fe_bond_mean': np.nan, 'fe_bond_max': np.nan})

    if coords.size == 0:
        return pd.Series({'fe_bond_min': np.nan, 'fe_bond_mean': np.nan, 'fe_bond_max': np.nan})
    dists = np.linalg.norm(coords - fe_coord.reshape(1, 3), axis=1)
    return pd.Series({'fe_bond_min': float(np.nanmin(dists)),
                      'fe_bond_mean': float(np.nanmean(dists)),
                      'fe_bond_max': float(np.nanmax(dists))})

# Apply to the dataframe (may be slow; consider vectorized approaches for large datasets)
if 'fe_x' in df.columns:
    fe_features = df.apply(compute_fe_bond_lengths, axis=1)
    df = pd.concat([df, fe_features], axis=1)
    print('Computed Fe bond-length features')
else:
    print('fe_x/fe_y/fe_z not found; skipping Fe bond-length computation')




---

## Part 4 — Data filtering and querying

Demonstrate typical data filters: restrict to a single oxidation state, remove very large complexes, or restrict by computed criteria.



In [ ]:

# Example: select Fe(2) complexes only
if 'fe_oxidation_state' in df.columns:
    df = df[df['fe_oxidation_state'] == 2].copy()
    print('Filtered to Fe(2). Remaining rows:', len(df))

# Example: remove extreme charge values
if 'charge' in df.columns:
    df = df[df['charge'].abs() <= 3].copy()
    print('Removed large charges. Remaining rows:', len(df))




**Mentor checkpoint 3**

- Confirm dataset selection and target column are correct for the group.
- Discuss any domain-specific filters (oxidation state, solvent, basis set) to apply before modeling.
- Decide whether to drop outliers flagged by `is_outlier_any` or to keep them and add robust models later.

Proceed only after confirmation.

---

### Exercise 1.1 — Quick data exploration (10 minutes)

1. Print the distribution (value counts) of a categorical column such as `ligand_type` or `solvent` if present.
2. Report the number of unique Fe centers (hint: group by a unique complex identifier).
3. Compute the fraction of rows that were flagged as outliers.

Write code to answer these questions and print short conclusions (1-2 lines each).

### Exercise 1.2 — NumPy operations on structure arrays (20 minutes)

1. Using the `parse_xyz_file` helper, read one structure file and compute the centroid of its atoms. Use NumPy vector operations.

Recall z-score normalization for a sample:

$$
z_i = \frac{x_i - \mu}{\sigma}
$$

and the sample covariance between X and Y:

$$
\mathrm{Cov}(X,Y) = \frac{1}{n-1}\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})
$$

2. Compute the z-scores for the Fe–ligand bond lengths you generated and compute the covariance between `fe_bond_mean` and `redox_potential`.

### Exercise 1.3 — Save the cleaned dataset (5 minutes)

Save `df` to `data_cleaned.csv` in the notebook folder. This file will be used by Notebook 2.



In [ ]:

df.to_csv('data_cleaned.csv', index=False)
print('Wrote data_cleaned.csv with', len(df), 'rows')




---

## Summary

| Task | Tool | Key functions |
|---|---:|---|
| Load data | pandas | `pd.read_csv`, `pd.read_json` |
| Inspect | pandas | `df.head`, `df.describe`, `df.dtypes` |
| Outlier detection | pandas, numpy | quantiles, boolean masks |
| Feature engineering | numpy, custom | `compute_fe_bond_lengths` |

---

# Notebook 2 — Exploratory Data Analysis & Visualization (Matplotlib & Seaborn)

**Duration:** 75 minutes  
**Audience:** Undergraduates with intermediate Python, introductory chemistry knowledge  
**Goal:** visualize target and feature distributions, identify relationships, and produce publication-quality diagnostic plots.

## Learning objectives

- Plot distributions and summary statistics of the target variable.
- Visualize pairwise relationships and compute correlation matrices (Pearson and Spearman).
- Use violin plots and 2D density plots to inspect conditional distributions.
- Save figures for use in reports and presentations.

---

## Part 1 — Distribution analysis

### 1.1 Load the cleaned data and imports



In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set(style='whitegrid', context='notebook')

df = pd.read_csv('data_cleaned.csv')
print('Loaded cleaned data with', len(df), 'rows')
display(df.head())




### 1.2 Target distribution and moments

Plot the distribution of the redox target and compute skewness and kurtosis. The sample skewness and (excess) kurtosis are defined as:

$$
\text{skew}(X) = \mathbb{E}\left[\left(\frac{X - \mu}{\sigma}\right)^{3}\right], \quad \text{kurt}(X) = \mathbb{E}\left[\left(\frac{X - \mu}{\sigma}\right)^{4}\right] - 3
$$



In [ ]:

if 'redox_potential' not in df.columns:
    raise RuntimeError('redox_potential column not found in cleaned data')

plt.figure(figsize=(6,4))
sns.histplot(df['redox_potential'], kde=True)
plt.title('Redox potential distribution')
plt.savefig('redox_distribution.png', dpi=150)
plt.show()

print('skewness:', df['redox_potential'].skew())
print('kurtosis (excess):', df['redox_potential'].kurt())




### 1.3 Feature distributions

Visualize a small set of numeric features with histograms and KDEs. This helps spot multimodality and heavy tails.



In [ ]:

num_cols = df.select_dtypes(include=["number"]).columns.tolist()
num_cols = [c for c in num_cols if c not in ['index']][:8]

def plot_feature_distributions(df, cols):
    n = len(cols)
    ncols = 4
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows))
    axes = np.atleast_1d(axes).flatten()
    for ax, c in zip(axes, cols):
        sns.histplot(df[c].dropna(), kde=True, ax=ax)
        ax.set_title(c)
    for ax in axes[len(cols):]:
        ax.set_visible(False)
    fig.tight_layout()
    fig.savefig('feature_distributions.png', dpi=150)
    plt.show()

plot_feature_distributions(df, num_cols[:8])




---

## Part 2 — Relationship analysis

### 2.1 Scatter plots and Pearson correlation

Pearson correlation coefficient between two variables is defined as:

$$
r_{XY} = \frac{\sum_{i} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i} (x_i - \bar{x})^2}\ \sqrt{\sum_{i} (y_i - \bar{y})^2}}
$$

Plot a selection of features against the target and overlay a linear regression fit.



In [ ]:

pairs = ['fe_bond_mean', 'homo_energy', 'lumo_energy']

pairs = [p for p in pairs if p in df.columns]
if not pairs:
    pairs = num_cols[:3]

fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs), 4), squeeze=False)
axes = np.atleast_1d(axes).flatten()
for ax, c in zip(axes, pairs):
    sns.regplot(x=c, y='redox_potential', data=df, ax=ax, scatter_kws={'s':10}, line_kws={'color':'red'})
    ax.set_title(f'{c} vs redox_potential')
for ax in axes[len(pairs):]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig('feature_vs_target_scatter.png', dpi=150)
plt.show()

for c in pairs:
    r = df[c].corr(df['redox_potential'])
    print(f'Pearson r ({c} vs redox):', r)




### 2.2 Pairplot for multi-feature relationships

A pairplot (scatter + KDE on diag) helps identify non-linear relationships and clusters.



In [ ]:

cols_for_pairplot = [c for c in ['fe_bond_mean', 'fe_bond_min', 'fe_bond_max', 'homo_energy', 'lumo_energy', 'redox_potential'] if c in df.columns]
if len(cols_for_pairplot) > 1:
    sns.pairplot(df[cols_for_pairplot].dropna(), diag_kind='kde', corner=True)
    plt.savefig('pairplot.png', dpi=150)
    plt.show()




---

## Part 3 — Correlation analysis

### 3.1 Correlation matrix heatmap

Compute Pearson correlation matrix and plot a heatmap.



In [ ]:

corr = df[num_cols].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Pearson correlation matrix')
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()




### 3.2 Spearman rank correlation

Spearman rank correlation is useful for monotonic but non-linear relationships. The rank correlation coefficient is:

$$
\rho = 1 - \frac{6 \sum_{i} d_i^2}{n(n^2 - 1)}\quad\text{where } d_i = \mathrm{rank}(x_i) - \mathrm{rank}(y_i)
$$

Compute and compare Spearman to Pearson for the target relationships.



In [ ]:

spearman = df[num_cols].corr(method='spearman')
print('Top correlations with redox (Pearson):')
print(corr['redox_potential'].abs().sort_values(ascending=False).head(10))
print('\nTop correlations with redox (Spearman):')
print(spearman['redox_potential'].abs().sort_values(ascending=False).head(10))




---

## Part 4 — Advanced visualizations

### 4.1 Violin plots by category

If a categorical variable exists (e.g., `ligand_type` or `solvent`), violin plots show conditional distributions of the target.



In [ ]:

cat = None
for c in df.select_dtypes(include=['object', 'category']).columns:
    if df[c].nunique() < 10:
        cat = c
        break

if cat:
    plt.figure(figsize=(8,4))
    sns.violinplot(x=cat, y='redox_potential', data=df)
    plt.xticks(rotation=45)
    plt.title(f'Redox by {cat}')
    plt.savefig('violin_plots.png', dpi=150)
    plt.show()
else:
    print('No low-cardinality categorical column found for violin plots')




### 4.2 2D density plot

A 2D kernel density estimate lets you visualize joint distributions and identify dense regions where models should focus their fit.



In [ ]:

if 'fe_bond_mean' in df.columns and 'homo_energy' in df.columns:
    plt.figure(figsize=(6,5))
    sns.kdeplot(x=df['fe_bond_mean'], y=df['homo_energy'], cmap='Blues', fill=True, thresh=0.05)
    plt.xlabel('fe_bond_mean')
    plt.ylabel('homo_energy')
    plt.title('2D density: fe_bond_mean vs homo_energy')
    plt.savefig('2d_density.png', dpi=150)
    plt.show()




**Mentor checkpoint 4**

- Confirm whether the observed correlations align with chemical intuition.
- Discuss which features are plausible inputs for classical baselines (RF, GPR) vs. GNN inputs.
- Decide on a shortlist of features to use in Day 3 baselines.

Proceed only after confirmation.

---

### Exercise 2.1 — Custom correlation heatmap (20 minutes)

Create a custom heatmap focusing on the top 10 features most correlated (absolute Pearson r) with `redox_potential`. Save it as `exercise_2_1_heatmap.png`.

Provide code that computes the top features, recomputes the correlation submatrix, and plots it with annotations.

### Exercise 2.2 — Feature relationship deep dive (25 minutes)

Pick one feature (for example `fe_bond_mean`) and explore polynomial relationships to the redox target. Fit polynomial models of degree 1..5 and report validation R² for each.

Coefficient of determination (R²):

$$
R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}
$$

Hint: Split the data into a train/validation split (e.g., 80/20) or use k-fold cross-validation to compare polynomial degrees fairly and avoid overfitting.

### Exercise 2.3 — Discussion (5 minutes)

Answer in Markdown: based on the EDA, what modeling approach would you try first and why? Consider model complexity, interpretability, and dataset size.

---

## Summary

| Visualization | Purpose | Library |
|---|---:|---|
| Histogram + KDE | Inspect target distribution, skewness | seaborn |
| Scatter + regplot | Inspect linear trends with target | seaborn |
| Pairplot | Multi-feature relationships | seaborn |
| Correlation heatmap | Global linear associations | seaborn, matplotlib |
| Violin / KDE / 2D density | Conditional distributions and joint density | seaborn |

---

## Preparation for Day 3

Day 3 will cover dimensionality reduction (PCA), clustering (K-Means), and classical ML baselines (Random Forest, Gaussian Process Regression). Make sure `data_cleaned.csv` exists and confirm the shortlist of features you want to try as baselines.